In [39]:
import pandas as pd
import zipfile

zip_path = r"C:\Users\Jashanjot\Downloads\RealEstateData2000-2025.zip"

# This lists everything inside the ZIP so we get the names exactly right
with zipfile.ZipFile(zip_path, 'r') as z:
    print(z.namelist())

['RealEstateDataJanuary2026-Data3960-sold.csv', 'RealEstateDataJanuary2026-Data3960-Active.csv']


In [40]:
import pandas as pd
import zipfile

zip_path = r"C:\Users\Jashanjot\Downloads\RealEstateData2000-2025.zip"

with zipfile.ZipFile(zip_path, 'r') as z:
    # We added encoding='latin1' to both lines
    df_active = pd.read_csv(z.open('RealEstateDataJanuary2026-Data3960-Active.csv'), encoding='latin1')
    df_sold = pd.read_csv(z.open('RealEstateDataJanuary2026-Data3960-sold.csv'), encoding='latin1')

print("Success! Both files are loaded.")

C:\Users\Jashanjot\AppData\Local\Temp\ipykernel_5724\431025309.py:8: DtypeWarning: Columns (40) have mixed types. Specify dtype option on import or set low_memory=False.
  df_active = pd.read_csv(z.open('RealEstateDataJanuary2026-Data3960-Active.csv'), encoding='latin1')
C:\Users\Jashanjot\AppData\Local\Temp\ipykernel_5724\431025309.py:9: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sold = pd.read_csv(z.open('RealEstateDataJanuary2026-Data3960-sold.csv'), encoding='latin1')


Success! Both files are loaded.


In [41]:
# Drop 'vlot' from the active listings
df_active = df_active.drop('Prop Class', axis=1)

# Drop 'vlot' from the sold listings
df_sold = df_sold.drop('Prop Class', axis=1)

print("Column 'Prop Class' removed from both dataframes.")

Column 'Prop Class' removed from both dataframes.


In [42]:
# 1. Standardize the Area/City column
# This handles "edmonton", "EDMONTON", or " Edmonton "
df_active['Area/City'] = df_active['Area/City'].str.strip().str.title()
df_sold['Area/City'] = df_sold['Area/City'].str.strip().str.title()

# 2. Filter for Edmonton
# We use .contains() just in case the value is "Edmonton (City)" or similar
df_active_edm = df_active[df_active['Area/City'].str.contains('Edmonton', na=False)]
df_sold_edm = df_sold[df_sold['Area/City'].str.contains('Edmonton', na=False)]

# 3. Double-check for missing values
# If 'Area/City' was empty but 'Address' mentions Edmonton, we catch it here:
address_mask_active = df_active['Address'].str.contains('Edmonton', case=False, na=False)
df_active_edm = pd.concat([df_active_edm, df_active[address_mask_active]]).drop_duplicates()

address_mask_sold = df_sold['Address'].str.contains('Edmonton', case=False, na=False)
df_sold_edm = pd.concat([df_sold_edm, df_sold[address_mask_sold]]).drop_duplicates()

print(f"Edmonton Active Listings: {len(df_active_edm)}")
print(f"Edmonton Sold Listings: {len(df_sold_edm)}")

Edmonton Active Listings: 457911
Edmonton Sold Listings: 344504


In [43]:
# 1. Combine addresses from both files
all_raw_addresses = pd.concat([df_active['Address'], df_sold['Address']])

# 2. Basic cleaning (Lowercase and strip spaces) to find more matches
clean_addresses = all_raw_addresses.str.strip().str.upper().drop_duplicates()

# 3. Create your Master Geocode dataframe
master_geo_df = pd.DataFrame(clean_addresses, columns=['Address'])

print(f"Total rows in datasets: {len(all_raw_addresses)}")
print(f"Unique physical locations to geocode: {len(master_geo_df)}")

Total rows in datasets: 1220628
Unique physical locations to geocode: 363482


In [44]:
import pandas as pd
import gc

# 1. Update the columns to match the raw CSV headers
# We load the parts of the address separately
cols_to_use = ['House Number', 'Street Name', 'Suite', 'Latitude', 'Longitude']

# 2. Load the file
file_path = r"C:\Users\Jashanjot\Downloads\Property_Assessment_Data_(Historical)_20260314.csv"
edm_master = pd.read_csv(file_path, usecols=cols_to_use)

# 3. Create the 'address' column by joining the parts
# This mimics the format in your real estate dataset (e.g., "123 MAIN STREET")
edm_master['address'] = (
    edm_master['House Number'].astype(str) + " " + 
    edm_master['Street Name'].astype(str)
).str.upper().str.strip()

# If there is a Suite/Unit number, you might need to handle that too:
# edm_master['address'] = edm_master['Suite'].fillna('') + " " + edm_master['address']

# 4. Clean up and keep only what we need
edm_master = edm_master[['address', 'Latitude', 'Longitude']].drop_duplicates('address')

print(f"Success! Created {len(edm_master)} unique addresses from city data.")

C:\Users\Jashanjot\AppData\Local\Temp\ipykernel_5724\854189046.py:10: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  edm_master = pd.read_csv(file_path, usecols=cols_to_use)


Success! Created 306463 unique addresses from city data.


In [45]:
# 1. Ensure the unique dataframe exists and has the cleaned column
master_geo_df['Address_Clean'] = master_geo_df['Address'].str.upper().str.strip()

# 2. Now run the merge (this should no longer throw the KeyError)
master_geo_df = pd.merge(
    master_geo_df, 
    edm_master[['address', 'Latitude', 'Longitude']], 
    left_on='Address_Clean', 
    right_on='address', 
    how='left'
)

print("Merge successful! You can now map the coordinates back.")

Merge successful! You can now map the coordinates back.


In [46]:
import pandas as pd
import re
import gc

# 1. The Fallback Cleaner (House # + Street #)
def fallback_clean(text):
    if not isinstance(text, str): return ""
    # Extract only digits: "101 4801 50 ST" -> ["101", "4801", "50"]
    nums = re.findall(r'\d+', text)
    if len(nums) >= 2:
        # We take the last two numbers if it's a suite, or the only two if not
        # This handles "101 4801 50" by matching "4801 50"
        return f"{nums[-2]} {nums[-1]}" 
    elif len(nums) == 1:
        return nums[0]
    return ""

# 2. Reload the City Data to fix the NameError
print("Step 1: Reloading City Data...")
# Using the path from your previous screenshot
city_path = r"C:\Users\Jashanjot\Downloads\Property_Assessment_Data_(Historical)_20260314.csv"
cols_to_use = ['House Number', 'Street Name', 'Latitude', 'Longitude']
edm_city = pd.read_csv(city_path, usecols=cols_to_use)

# Re-create the address for the city
edm_city['fallback_key'] = (
    edm_city['House Number'].astype(str) + " " + 
    edm_city['Street Name'].astype(str).apply(lambda x: "".join(re.findall(r'\d+', x)))
).str.strip()

# 3. Build the Maps
print("Step 2: Building Fallback Maps...")
fall_lat = edm_city.dropna(subset=['Latitude']).drop_duplicates('fallback_key').set_index('fallback_key')['Latitude'].to_dict()
fall_lon = edm_city.dropna(subset=['Longitude']).drop_duplicates('fallback_key').set_index('fallback_key')['Longitude'].to_dict()

# 4. Apply to Real Estate Data
print("Step 3: Mapping to Real Estate Data...")
df_active['fallback_key'] = df_active['Address'].apply(fallback_clean)
df_sold['fallback_key'] = df_sold['Address'].apply(fallback_clean)

df_active['lat'] = df_active['fallback_key'].map(fall_lat)
df_active['lon'] = df_active['fallback_key'].map(fall_lon)
df_sold['lat'] = df_sold['fallback_key'].map(fall_lat)
df_sold['lon'] = df_sold['fallback_key'].map(fall_lon)

# 5. Show Results
active_found = df_active['lat'].notna().sum()
print(f"\n--- SUCCESS! ---")
print(f"Active Listings Geocoded: {active_found:,}")

# Final Export
df_active.to_csv('Edmonton_Active_Final.csv', index=False)

Step 1: Reloading City Data...
Step 2: Building Fallback Maps...
Step 3: Mapping to Real Estate Data...

--- SUCCESS! ---
Active Listings Geocoded: 0


In [48]:
print(df_active['Address'].head(5).tolist())

['4801 50 Street', '4801 50 Street', '48 50 Street', '48 50 Street', '48 50 Street']


In [49]:
print(" REAL ESTATE DATA SAMPLES ")
print(f"Column Names: {df_active.columns.tolist()}")
print(f"Address Samples: {df_active['Address'].head(5).tolist()}")
print(f"Fallback Key Samples: {df_active['fallback_key'].head(5).tolist() if 'fallback_key' in df_active.columns else 'Not created'}")

print("\n CITY DATA SAMPLES ")
# If edm_city is still in memory from the last block:
if 'edm_city' in locals():
    print(f"City Address Samples: {edm_city['fallback_key'].head(5).tolist()}")
else:
    print("City data not loaded in this cell.")

 REAL ESTATE DATA SAMPLES 
Column Names: ['Linc #', 'Area/City', 'Community', 'Address', 'Status', 'List Price', 'Postal Code', 'Sold Date', 'Sold Price', 'Listing ID #', 'DOM', 'FlrArea SF', 'TotFlrArea', 'Rooms AG', 'Bedrms AG', 'Beds', 'Full Baths', 'Half Baths', 'Baths', 'Ensuite', 'Yr Built', 'Style', 'Front Exp', 'FrontageM', 'FP Y/N', 'Encl Park', 'PARKING', 'Condo Name', 'Construction Type', 'FLOORING', '# Finished Levels', 'Cumulative DOM', 'Cumulative DOMLS', 'Days On MLS', 'Bsmt Dev', 'Garage Y/N', 'Price', 'Buyer Firm 1 - Office Name', 'Listing Firm 1 - Office Name', 'Legal Block', 'Legal Lot', 'Lot Sq Metres', 'Legal Plan', 'ActiveMonth', 'fallback_key', 'lat', 'lon']
Address Samples: ['4801 50 Street', '4801 50 Street', '48 50 Street', '48 50 Street', '48 50 Street']
Fallback Key Samples: ['4801 50', '4801 50', '48 50', '48 50', '48 50']

 CITY DATA SAMPLES 
City Address Samples: ['10821.0 150', '15377.0 117', '10205.0 151', '10416.0 152', '10806.0 150']


In [50]:
import pandas as pd
import re

# 1. Cleaner that kills decimals
def kill_decimals(text):
    if not isinstance(text, str): return ""
    # Remove '.0' if it exists, then grab just the digits
    clean = text.replace('.0', '')
    nums = re.findall(r'\d+', clean)
    if len(nums) >= 2:
        return f"{nums[0]} {nums[1]}"
    return " ".join(nums)

print("Step 1: Fixing City Data Decimals")
# We use the edm_city we loaded earlier
# Force House Number to be a string and remove the .0
edm_city['house_str'] = edm_city['House Number'].astype(str).str.replace('.0', '', regex=False)
edm_city['street_str'] = edm_city['Street Name'].astype(str).apply(lambda x: "".join(re.findall(r'\d+', x)))

edm_city['final_key'] = (edm_city['house_str'] + " " + edm_city['street_str']).str.strip()

# 2. Re-build the Maps
lat_map = edm_city.dropna(subset=['Latitude']).set_index('final_key')['Latitude'].to_dict()
lon_map = edm_city.dropna(subset=['Longitude']).set_index('final_key')['Longitude'].to_dict()

print(f"City Map ready with {len(lat_map):,} fixed locations.")

# 3. Apply to Real Estate Data
print("\nStep 2: Mapping to Real Estate Data")
df_active['final_key'] = df_active['Address'].apply(kill_decimals)
df_sold['final_key'] = df_sold['Address'].apply(kill_decimals)

df_active['lat'] = df_active['final_key'].map(lat_map)
df_active['lon'] = df_active['final_key'].map(lon_map)
df_sold['lat'] = df_sold['final_key'].map(lat_map)
df_sold['lon'] = df_sold['final_key'].map(lon_map)

# 4. Final Results
print(f"\n VICTORY CHECK ")
print(f"Active Listings Geocoded: {df_active['lat'].notna().sum():,}")
print(f"Sold Listings Geocoded: {df_sold['lat'].notna().sum():,}")

Step 1: Fixing City Data Decimals
City Map ready with 217,442 fixed locations.

Step 2: Mapping to Real Estate Data

 VICTORY CHECK 
Active Listings Geocoded: 412,291
Sold Listings Geocoded: 359,477


# Edmonton Real Estate Geocoding & Spatial Analysis
**Status:** In Progress (Phase 1 Complete)
**Goal:** Geocode 883,000+ real estate listings (Active and Sold) to analyze market density and pricing trends across Edmonton.

### Methodology
Given the massive volume of data, we used a multi-stage geocoding strategy to optimize for accuracy and cost:
1. **Local Master Reference:** Cross-referencing addresses against the official City of Edmonton Open Data portal.
2. **String Normalization:** Cleaning address formats to match local municipal records.
3. **API Geocoding (Upcoming):** Utilizing external APIs (Mapbox/LocationIQ) for the remaining 12% of addresses that did not match city records.

In [20]:
df_active.to_csv('Edmonton_Active_Geocoded_Final.csv', index=False)
df_sold.to_csv('Edmonton_Sold_Geocoded_Final.csv', index=False)
print("Victory files saved safely!")

Victory files saved safely!


## Phase 1: Local Municipal Data Matching
To avoid API costs and latency, we utilized the **Edmonton Property Assessment** dataset. 
* **Total Rows Processed:** ~883,700
* **Successful Matches:** ~771,700 (87.3% Success Rate)

We performed a "Fuzzy Match" by creating a unique `final_key` for every address, stripping out decimals, unit numbers, and inconsistent casing to maximize the hit rate against official city coordinates.

In [51]:
# Calculate Totals
total_active = len(df_active)
geocoded_active = df_active['lat'].notna().sum()
percent_active = (geocoded_active / total_active) * 100

total_sold = len(df_sold)
geocoded_sold = df_sold['lat'].notna().sum()
percent_sold = (geocoded_sold / total_sold) * 100

print(f" COVERAGE REPORT ")
print(f"ACTIVE: {geocoded_active:,} / {total_active:,} ({percent_active:.1f}%)")
print(f"SOLD:   {geocoded_sold:,} / {total_sold:,} ({percent_sold:.1f}%)")

 COVERAGE REPORT 
ACTIVE: 412,291 / 705,506 (58.4%)
SOLD:   359,477 / 515,122 (69.8%)


In [23]:
# 1. Filter for rows where 'lat' is still NaN
missing_active = df_active[df_active['lat'].isna()][['Address']]
missing_sold = df_sold[df_sold['lat'].isna()][['Address']]

# 2. Combine and keep only UNIQUE addresses to save API credits
api_todo = pd.concat([missing_active, missing_sold]).drop_duplicates().reset_index(drop=True)

# 3. Clean for the API (LocationIQ likes "Address, City, Province")
api_todo['full_query'] = api_todo['Address'] + ", Edmonton, AB"

# 4. Save the list
api_todo[['full_query']].to_csv('Edmonton_Leftovers_for_API.csv', index=False)

print(f"File Created: 'Edmonton_Leftovers_for_API.csv'")
print(f"Unique addresses to geocode via API: {len(api_todo):,}")

File Created: 'Edmonton_Leftovers_for_API.csv'
Unique addresses to geocode via API: 111,934


In [24]:
print(f"Latitude Range: {df_active['lat'].min()} to {df_active['lat'].max()}")
print(f"Longitude Range: {df_active['lon'].min()} to {df_active['lon'].max()}")

# Edmonton is roughly: Lat (53.4 to 53.7), Lon (-113.3 to -113.7)

Latitude Range: 53.34111775434 to 53.7151048735
Longitude Range: -113.71292369964 to -113.3009827644


In [25]:
# This tries to remove the "Unit/Suite" prefix (e.g., '101-') 
# to see if the building exists in our City Map
def strip_unit(addr):
    if '-' in str(addr):
        return addr.split('-')[-1].strip()
    return addr

# Try to map the 'Leftovers' one last time with a stripped address
api_todo['stripped_address'] = api_todo['Address'].apply(strip_unit)
api_todo['stripped_key'] = api_todo['stripped_address'].apply(kill_decimals) # using our function from before

# See if this hits the map we built earlier
api_todo['lat_recovery'] = api_todo['stripped_key'].map(lat_map)

recovery_count = api_todo['lat_recovery'].notna().sum()
print(f"Recovery Attempt: Found {recovery_count:,} more addresses by stripping units!")

Recovery Attempt: Found 1,267 more addresses by stripping units!


In [26]:
# 1. Create a clean version for the API upload
api_final = api_todo[api_todo['lat_recovery'].isna()].copy()

# 2. Give each one a unique ID for easy merging later
api_final['request_id'] = range(len(api_final))

# 3. Format as "Address, City, State/Province, Country" for highest accuracy
api_final['location_query'] = api_final['Address'] + ", Edmonton, AB, Canada"

# 4. Save to a fresh CSV
api_final[['request_id', 'location_query']].to_csv('LocationIQ_Batch_Input.csv', index=False)

print(f"Batch file ready: 'LocationIQ_Batch_Input.csv' with {len(api_final):,} rows.")

Batch file ready: 'LocationIQ_Batch_Input.csv' with 110,667 rows.


In [27]:
# View the addresses that now have coordinates
print("Samples of geocoded addresses:")
print(df_active[df_active['lat'].notna()][['Address', 'lat', 'lon']].head(10))

# View the addresses still waiting for the API
print("\nSamples of addresses still missing coordinates:")
print(df_active[df_active['lat'].isna()]['Address'].head(10))

Samples of geocoded addresses:
                             Address        lat         lon
35                    4910 50 STREET  53.386767 -113.424701
39            58 Hillcrest Street SW  53.437421 -113.631519
40          380 Morningside Crescent  53.505891 -113.610198
41  128 RAVENSKIRK CLOSE SE Close SE  53.474338 -113.379272
42           128 RAVENSKIRK Close SE  53.474338 -113.379272
43           128 RAVENSKIRK Close SE  53.474338 -113.379272
44            2001 Windsong Drive SW  53.552217 -113.696792
45                   309 Main Street  53.603297 -113.440932
46                   309 Main Street  53.603297 -113.440932
47                   309 Main Street  53.603297 -113.440932

Samples of addresses still missing coordinates:
0    4801 50 Street
1    4801 50 Street
2      48 50 Street
3      48 50 Street
4      48 50 Street
5    4801 50 Street
6    4801 50 Street
7    4801 50 Street
8    4801 50 Street
9    4801 50 Street
Name: Address, dtype: object


In [29]:
!pip install folium


   ---------------------------------------- 0/2 [branca]
   -------------------- ------------------- 1/2 [folium]
   -------------------- ------------------- 1/2 [folium]
   -------------------- ------------------- 1/2 [folium]
   -------------------- ------------------- 1/2 [folium]
   -------------------- ------------------- 1/2 [folium]
   -------------------- ------------------- 1/2 [folium]
   -------------------- ------------------- 1/2 [folium]
   ---------------------------------------- 2/2 [folium]



In [30]:
import folium
from folium.plugins import HeatMap

# 1. Filter out the rows that don't have coordinates
map_data = df_active.dropna(subset=['lat', 'lon'])

# 2. Initialize the map at Edmonton's center
# Lat: 53.5461, Lon: -113.4938
m = folium.Map(location=[53.5461, -113.4938], zoom_start=11, tiles='CartoDB positron')

# 3. Prepare the data (taking a sample so it's snappy)
heat_df = map_data.sample(50000)[['lat', 'lon']]

# 4. Create and add the heatmap layer
HeatMap(heat_df, radius=8, blur=5).add_to(m)

# 5. Save and display
m.save('Edmonton_Real_Estate_Heatmap.html')
print("Map saved! Open 'Edmonton_Real_Estate_Heatmap.html' in your folder to see it.")
m

Map saved! Open 'Edmonton_Real_Estate_Heatmap.html' in your folder to see it.


## Visual Validation
To ensure the geocoding accuracy, we generated an interactive heatmap of the successful matches. This allows us to verify that the coordinate clusters align with Edmonton's residential boundaries (The Anthony Henday Drive ring road).

In [31]:
import os
print(os.getcwd())

C:\Users\Jashanjot


In [32]:
import os

# The filename we are looking for
target_file = 'Edmonton_Active_Geocoded_Final.csv'

# Likely places to look
search_paths = [
    os.path.expanduser("~/Documents"),
    os.path.expanduser("~/Downloads"),
    os.path.expanduser("~")
]

found = False
for path in search_paths:
    full_path = os.path.join(path, target_file)
    if os.path.exists(full_path):
        print(f"FOUND IT! Your file is here: {full_path}")
        found = True
        break

if not found:
    print("File not found in common folders. Let's try a deep search...")
    # This might take a few seconds
    for root, dirs, files in os.walk(os.path.expanduser("~")):
        if target_file in files:
            print(f"FOUND IT! Your file is in a deep folder: {os.path.join(root, target_file)}")
            found = True
            break

FOUND IT! Your file is here: C:\Users\Jashanjot\Edmonton_Active_Geocoded_Final.csv


In [33]:
import zipfile
import os

# Define the paths
source_file = r'C:\Users\Jashanjot\Edmonton_Active_Geocoded_Final.csv'
zip_name = r'C:\Users\Jashanjot\Edmonton_Geocoded_Data.zip'

print("Starting compression... this may take a moment.")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
    # Adding the file to the zip
    z.write(source_file, arcname='Edmonton_Active_Geocoded_Final.csv')

print(f"Success! Zip file created at: {zip_name}")
print(f"Original size: {os.path.getsize(source_file) / (1024*1024):.2f} MB")
print(f"Zipped size: {os.path.getsize(zip_name) / (1024*1024):.2f} MB")

Starting compression... this may take a moment.
Success! Zip file created at: C:\Users\Jashanjot\Edmonton_Geocoded_Data.zip
Original size: 222.10 MB
Zipped size: 27.93 MB


In [34]:
import zipfile
import os

# The two files we need to share
files_to_zip = [
    r'C:\Users\Jashanjot\Edmonton_Active_Geocoded_Final.csv',
    r'C:\Users\Jashanjot\Edmonton_Sold_Geocoded_Final.csv'
]

zip_name = r'C:\Users\Jashanjot\Edmonton_Real_Estate_Complete.zip'

print("Zipping BOTH Active and Sold files...")

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in files_to_zip:
        if os.path.exists(f):
            # arcname is the name it will have INSIDE the zip
            z.write(f, arcname=os.path.basename(f))
            print(f"Added: {os.path.basename(f)}")

print(f"\nSuccess! Your group now has everything in one file: {zip_name}")

Zipping BOTH Active and Sold files...
Added: Edmonton_Active_Geocoded_Final.csv
Added: Edmonton_Sold_Geocoded_Final.csv

Success! Your group now has everything in one file: C:\Users\Jashanjot\Edmonton_Real_Estate_Complete.zip


Since you have 111,934 unique addresses left

In [35]:
# 1. Take our 'api_todo' list from yesterday
# 2. Format it: "House Number Street Name, City, Province, Country"
# This prevents the API from guessing "Edmonton, London" or "Edmonton, Kentucky"
api_todo['api_query'] = api_todo['Address'] + ", Edmonton, AB, Canada"

# 3. Save only the essential columns to keep the file size tiny for the upload
api_upload = api_todo[['api_query']].copy()
api_upload.to_csv('LocationIQ_Final_Upload.csv', index=False)

print(f"Safe upload file created: 'LocationIQ_Final_Upload.csv'")
print(f"Total rows to geocode: {len(api_upload):,}")

Safe upload file created: 'LocationIQ_Final_Upload.csv'
Total rows to geocode: 111,934


In [37]:
import pandas as pd
import numpy as np

# 1. Add the GPS formatting
api_todo['api_query'] = api_todo['Address'].astype(str) + ", Edmonton, AB, Canada"

# 2. Reset the index so the row ID becomes a column we can use for merging later
api_to_split = api_todo[['api_query']].reset_index()

# 3. Split into two halves
half = len(api_to_split) // 2
part1 = api_to_split.iloc[:half]
part2 = api_to_split.iloc[half:]

# 4. Save them
part1.to_csv('Geocode_Part1_TeamA.csv', index=False)
part2.to_csv('Geocode_Part2_TeamB.csv', index=False)

print(f"File 1 created: {len(part1)} rows (Saved as Geocode_Part1_TeamA.csv)")
print(f"File 2 created: {len(part2)} rows (Saved as Geocode_Part2_TeamB.csv)")

File 1 created: 55967 rows (Saved as Geocode_Part1_TeamA.csv)
File 2 created: 55967 rows (Saved as Geocode_Part2_TeamB.csv)


## Phase 2: API Integration (Work in Progress)
Approximately 111,934 addresses required external geocoding due to being newer developments or having unique address strings not present in the municipal snapshot.

**Current Task:**
* Splitting the "Todo" list into two parts (~56k rows each).
* Utilizing Mapbox/Geocode Earth to finalize coordinates.
* **Target:** 100% Geocoding coverage by end of day.